In [61]:
%pip install pymupdf pillow easyocr
%pip install torch torchvision torchaudio

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 2.9/2.9 MB 30.7 MB/s  0:00:00
   ---------------------------------------- 0.0/40.1 MB ? eta -:--:--
   ------- -------------------------------- 7.6/40.1 MB 37.5 MB/s eta 0:00:01
   --------------- ------------------------ 15.7/40.1 MB 37.5 MB/s eta 0:00:01
   ----------------------- ---------------- 23.6/40.1 MB 37.6 MB/s eta 0:00:01
   ------------------------------- -------- 31.7/40.1 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------  39.3/40.1 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------- 40.1/40.1 MB 36.1 MB/s  0:00:01
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ------------------------- -------------- 7.6/11.9 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------- 11.9/11.9 MB 35.8 MB/s  0:00:00
   ------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 1. Data Loading & Preprocessing
Brief: This section loads NDA documents and preprocesses them into clauses.

In [129]:
import fitz  # PyMuPDF
import io
import numpy as np
import easyocr
from PIL import Image


# Initialize OCR reader once per session (downloads model files on first run)
ocr_reader = easyocr.Reader(["en"], gpu=torch.cuda.is_available())


def extract_text_from_pdf(file_path, zoom=2.0):
    """Extract embedded PDF text and fallback to EasyOCR for image-only pages."""
    doc = fitz.open(file_path)
    text_parts = []

    for page in doc:
        embedded_text = page.get_text().strip()
        if embedded_text:
            text_parts.append(embedded_text)
            continue

        # OCR fallback for scanned/image-only pages
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        img_np = np.array(img)

        # detail=0 returns text strings only, paragraph=True merges nearby lines
        ocr_lines = ocr_reader.readtext(img_np, detail=0, paragraph=True)
        ocr_text = "\n".join(line.strip() for line in ocr_lines if line and line.strip())
        text_parts.append(ocr_text)

    return "\n".join(part for part in text_parts if part)


# Load files
nda1_text = extract_text_from_pdf("NDA-1.pdf")
nda2_text = extract_text_from_pdf("NDA-1_ALT.pdf")

# Preview
#print(nda1_text[:1000])
print(nda2_text[:1000])

Classroom Use of Confidential Information 
Non-Disclosure Agreement (NDA) Template Package 
 
for Course Instructors 
Do not use the attached template if students 
are receiving third party proprietary 
confidential information: 
 ...as part of a student placement, internship, 
co-curricular or other off-site activity. 
Contact the Office of the Vice-Provost, 
Students for applicable templates and 
additional information related to these 
activities; or 
 ...to conduct research under the supervision 
of a principal investigator. Contact the 
Innovations & Partnerships Office for 
applicable templates and additional 
information related to supervised research. 
 
Use the attached template if: 
□ You have determined that receiving 
confidential information from a third party 
for classroom use would be highly beneficial 
to your students enrolled in the academic 
course you are teaching; 
□ You have obtained approval from your 
academic unit/division head, for you and 
your students to r

In [130]:
import re

def clean_text(text):
    # remove weird line breaks
    text = text.replace("\n", " ")
    
    # remove extra spaces
    text = re.sub(r"\s+", " ", text)
    
    # remove weird symbols
    text = re.sub(r"[^\x00-\x7F]+", "", text)
    
    return text.strip()

nda1_clean = clean_text(nda1_text)
nda2_clean = clean_text(nda2_text)

print(nda1_clean[:1000])
print(nda2_clean[:1000])

Classroom Use of Confidential Information Non-Disclosure Agreement (NDA) Template Package  for Course Instructors Do not use the attached template if students are receiving third party proprietary confidential information:  as part of a student placement, internship, co-curricular or other off-site activity. Contact the Office of the Vice-Provost, Students for applicable templates and additional information related to these activities; or  to conduct research under the supervision of a principal investigator. Contact the Innovations & Partnerships Office for applicable templates and additional information related to supervised research. Use the attached template if:  You have determined that receiving confidential information from a third party for classroom use would be highly beneficial to your students enrolled in the academic course you are teaching;  You have obtained approval from your academic unit/division head, for you and your students to receive confidential information from

In [131]:
def split_into_clauses(text, min_len=50):
    """Split text into clause-like chunks with OCR-friendly fallbacks."""
    cleaned = text.strip()
    if not cleaned:
        return []

    # Insert split hints for common section/list markers that OCR often flattens.
    normalized = re.sub(r"\s+(?=(?:\d+\.|[a-z]\)|Instructions:|APPENDIX\s+[A-Z]|Page\s+\d+\s+of\s+\d+))", "\n", cleaned)

    # First pass: split by explicit markers.
    chunks = re.split(r"\n+", normalized)
    chunks = [c.strip() for c in chunks if len(c.strip()) > min_len]

    # If still too coarse, fallback to sentence-group chunking.
    if len(chunks) <= 6:
        sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z\[])", cleaned)
        sentences = [s.strip() for s in sentences if s.strip()]

        grouped = []
        current = []
        current_len = 0
        for sentence in sentences:
            current.append(sentence)
            current_len += len(sentence)
            if current_len >= 170:
                grouped.append(" ".join(current))
                current = []
                current_len = 0
        if current:
            grouped.append(" ".join(current))

        fallback = [c.strip() for c in grouped if len(c.strip()) > min_len]
        if len(fallback) > len(chunks):
            return fallback

    return chunks

nda1_clauses = split_into_clauses(nda1_clean)
nda2_clauses = split_into_clauses(nda2_clean)

print("Number of clauses(1):", len(nda1_clauses))
print("\nExample clause:\n", nda1_clauses[0])

print("Number of clauses(2):", len(nda2_clauses))
print("\nExample clause:\n", nda2_clauses[0])

Number of clauses(1): 29

Example clause:
 Classroom Use of Confidential Information Non-Disclosure Agreement (NDA) Template Package  for Course Instructors Do not use the attached template if students are receiving third party proprietary confidential information:  as part of a student placement, internship, co-curricular or other off-site activity. Contact the Office of the Vice-Provost, Students for applicable templates and additional information related to these activities; or  to conduct research under the supervision of a principal investigator. Contact the Innovations & Partnerships Office for applicable templates and additional information related to supervised research. Use the attached template if:  You have determined that receiving confidential information from a third party for classroom use would be highly beneficial to your students enrolled in the academic course you are teaching;  You have obtained approval from your academic unit/division head, for you and your studen

In [93]:
%pip uninstall -y transformers
%pip install "transformers[sentencepiece]" --upgrade --force-reinstall

Note: you may need to restart the kernel to use updated packages.


Defaulting to user installation because normal site-packages is not writeable
  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached packaging-26.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.4.4-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.24.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl.metadata (10 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\alanc\\AppData\\Roaming\\Python\\Python313\\site-packages\\transformers\\models\\metaclip_2\\__init__.py'
Check the permissions.


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [94]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

2.12.0.dev20260314+cu128
GPU available: True


In [132]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

def summarize_text(text):
    input_text = "summarize: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=20
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [96]:
sample_clause = nda1_clauses[1][:500]

print("ORIGINAL:\n", sample_clause)
print("\nSUMMARY:\n", summarize_text(sample_clause))

ORIGINAL:
 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.

SUMMARY:
 academic divisions guidelines are designed to ensure that the academic unit/division head with signing authority is identified.


# 2. Feature 1: Summarization

## Baseline
Uses T5-small model for basic summarization.

## Improved
Uses prompt engineering and beam search to improve output quality.

## Comparison
Outputs from both approaches are compared.

In [133]:
def summarize_text_improved(text):
    input_text = "summarize key points: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=80,
        min_length=25,
        num_beams=4,           # better quality
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [134]:
print("BASELINE:\n", summarize_text(sample_clause))
print("\nIMPROVED:\n", summarize_text_improved(sample_clause))

BASELINE:
 academic divisions guidelines are designed to ensure that the academic unit/division head with signing authority is identified.

IMPROVED:
 the academic unit/division head with signing authority is identified. the academic unit/division head is not typically authorized to execute an NDA.


# 3. Feature 2: Risk Detection

## Baseline (Zero-shot)
Uses a pre-trained model to classify clauses as high or low risk.

## Improved (Logistic Regression)
Uses a trained machine learning model on labeled data.

## Comparison
Predictions from both models are compared.

In [135]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [136]:
labels = ["high risk", "low risk"]

result = classifier(sample_clause, candidate_labels=labels)

print(result)

{'sequence': '1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.', 'labels': ['high risk', 'low risk'], 'scores': [0.8147276043891907, 0.18527239561080933]}


In [137]:
data = [
    ("Confidential information must not be disclosed to third parties", "high risk"),
    ("Recipient must protect confidential data", "high risk"),
    ("All confidential information must be returned after termination", "high risk"),
    ("This agreement lasts for two years", "low risk"),
    ("The agreement is governed by local law", "low risk"),
    ("Students may use the information only for coursework", "low risk"),
]

texts = [x[0] for x in data]
labels = [x[1] for x in data]

In [38]:
%pip install scikit-learn

'c:\Program' is not recognized as an internal or external command,
operable program or batch file.


In [138]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

In [139]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression()
model_lr.fit(X, labels)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [140]:
test_X = vectorizer.transform([sample_clause])
prediction = model_lr.predict(test_X)

print("Prediction:", prediction[0])

Prediction: low risk


In [141]:
for clause in nda1_clauses[:5]:
    print("\nCLAUSE:\n", clause[:200])
    
    # Zero-shot
    result = classifier(clause, candidate_labels=["high risk", "low risk"])
    zero_shot_pred = result["labels"][0]
    
    # Logistic regression
    test_X = vectorizer.transform([clause])
    lr_pred = model_lr.predict(test_X)[0]
    
    print("Zero-shot:", zero_shot_pred)
    print("Logistic Regression:", lr_pred)


CLAUSE:
 Classroom Use of Confidential Information Non-Disclosure Agreement (NDA) Template Package  for Course Instructors Do not use the attached template if students are receiving third party proprietary con
Zero-shot: high risk
Logistic Regression: low risk

CLAUSE:
 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are 
Zero-shot: high risk
Logistic Regression: low risk

CLAUSE:
 2. Customize and complete all areas noted with an "INSERT...", which require content to be inserted by University of Toronto faculty or staff, as appopriate.
Zero-shot: high risk
Logistic Regression: high risk

CLAUSE:
 3. Notify the authorized signatory if changes have been made to this template by the third party prior to execution because proposed amendments require legal review.
Zero-shot: high risk
Logistic Regression: low risk

CLAUSE:
 4. Once the NDA is 

# 4. Feature 3: Jargon Explanation

## Baseline
Simple prompt-based explanation.

## Improved
Enhanced prompt and decoding for better clarity.

## Comparison
Outputs are compared to evaluate improvement.

In [142]:
def explain_clause_v1(text):
    input_text = "explain in simple terms: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=100,
        min_length=30
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [143]:
print("ORIGINAL:\n", sample_clause)
print("\nBASELINE EXPLANATION:\n", explain_clause_v1(sample_clause))

ORIGINAL:
 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.

BASELINE EXPLANATION:
 : 1. Consult your academic divisions guidelines with respect to the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are not typically authorized to execute an NDA.


In [144]:
def explain_clause_v2(text):
    input_text = "rewrite this legal sentence in very simple plain English for a non-lawyer: " + text
    
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    outputs = model.generate(
        inputs["input_ids"],
        max_length=120,
        min_length=40,
        num_beams=5,
        early_stopping=True
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [145]:
print("\nIMPROVED EXPLANATION:\n", explain_clause_v2(sample_clause))


IMPROVED EXPLANATION:
 1. Consult your academic divisions guidelines regarding the execution of contracts to ensure that the academic unit/division head with signing authority is identified, as course instructors are typically not typically authorized to execute an NDA.


# 5. Feature 4: Version Comparison and New Risk Detection

## Goal
Compare two versions of an NDA, highlight clause-level changes, and flag newly introduced risks.

## Approach
- Match clauses across versions with TF-IDF cosine similarity.
- Categorize changes as added, removed, or modified.
- Run risk classification on changed clauses and identify risk increases.
- Print a compact review report for legal/compliance triage.

In [149]:
from difflib import unified_diff
from sklearn.metrics.pairwise import cosine_similarity


def predict_risk_label(clause):
    """Use the existing zero-shot classifier for stable risk labels."""
    result = classifier(clause, candidate_labels=["high risk", "low risk"])
    return result["labels"][0], float(result["scores"][0])


def compare_clause_sets(
    old_clauses,
    new_clauses,
    modified_threshold=0.58,
    candidate_floor=0.35,
    structural_threshold=0.42,
):
    """
    Match clauses one-to-one using a broader candidate pool.
    Returns: added, removed, modified, unchanged, structural_changes, diagnostics
    """
    if not old_clauses and not new_clauses:
        return [], [], [], [], [], {"matched_pairs": 0, "candidate_pairs": 0}
    if not old_clauses:
        return new_clauses, [], [], [], [], {"matched_pairs": 0, "candidate_pairs": 0}
    if not new_clauses:
        return [], old_clauses, [], [], [], {"matched_pairs": 0, "candidate_pairs": 0}

    vec = TfidfVectorizer(ngram_range=(1, 2))
    old_X = vec.fit_transform(old_clauses)
    new_X = vec.transform(new_clauses)

    sim = cosine_similarity(new_X, old_X)  # rows: new, cols: old

    # Keep all viable pairings above a floor, plus each row's best match.
    candidate_pairs = []
    for new_idx in range(sim.shape[0]):
        row = sim[new_idx]
        best_old_idx = int(row.argmax())
        best_score = float(row[best_old_idx])
        candidate_pairs.append((best_score, new_idx, best_old_idx))

        for old_idx, score in enumerate(row):
            score = float(score)
            if score >= candidate_floor and old_idx != best_old_idx:
                candidate_pairs.append((score, new_idx, old_idx))

    candidate_pairs.sort(reverse=True, key=lambda x: x[0])

    matched_new = set()
    matched_old = set()
    modified = []
    unchanged = []

    for score, new_idx, old_idx in candidate_pairs:
        if new_idx in matched_new or old_idx in matched_old:
            continue
        if score < modified_threshold:
            continue

        matched_new.add(new_idx)
        matched_old.add(old_idx)

        old_clause = old_clauses[old_idx]
        new_clause = new_clauses[new_idx]
        if old_clause.strip() == new_clause.strip():
            unchanged.append({
                "old_clause": old_clause,
                "new_clause": new_clause,
                "similarity": score,
            })
        else:
            modified.append({
                "old_clause": old_clause,
                "new_clause": new_clause,
                "similarity": score,
            })

    unmatched_new_idx = [i for i in range(len(new_clauses)) if i not in matched_new]
    unmatched_old_idx = [i for i in range(len(old_clauses)) if i not in matched_old]

    # Structural changes: likely merge/split/reformat where similarity is moderate,
    # but not high enough to qualify as a direct one-to-one modified match.
    structural_changes = []
    structural_new = set()
    structural_old = set()

    for new_idx in unmatched_new_idx:
        links = []
        for old_idx in unmatched_old_idx:
            score = float(sim[new_idx, old_idx])
            if score >= structural_threshold:
                links.append({"old_idx": old_idx, "similarity": score})

        if links:
            links = sorted(links, key=lambda x: x["similarity"], reverse=True)[:3]
            structural_changes.append(
                {
                    "new_clause": new_clauses[new_idx],
                    "linked_old_clauses": [
                        {"old_clause": old_clauses[item["old_idx"]], "similarity": item["similarity"]}
                        for item in links
                    ],
                }
            )
            structural_new.add(new_idx)
            for item in links:
                structural_old.add(item["old_idx"])

    added = [new_clauses[i] for i in unmatched_new_idx if i not in structural_new]
    removed = [old_clauses[i] for i in unmatched_old_idx if i not in structural_old]

    diagnostics = {
        "matched_pairs": len(matched_new),
        "candidate_pairs": len(candidate_pairs),
        "threshold": modified_threshold,
        "candidate_floor": candidate_floor,
        "structural_threshold": structural_threshold,
    }

    return added, removed, modified, unchanged, structural_changes, diagnostics


def clause_word_diff(old_clause, new_clause, context_lines=1):
    old_words = old_clause.split()
    new_words = new_clause.split()
    diff_lines = list(unified_diff(old_words, new_words, fromfile="old", tofile="new", lineterm="", n=context_lines))
    return "\n".join(diff_lines)

In [150]:
def analyze_version_changes(
    old_clauses,
    new_clauses,
    modified_threshold=0.58,
    candidate_floor=0.35,
    structural_threshold=0.42,
):
    added, removed, modified, unchanged, structural_changes, diagnostics = compare_clause_sets(
        old_clauses,
        new_clauses,
        modified_threshold=modified_threshold,
        candidate_floor=candidate_floor,
        structural_threshold=structural_threshold,
    )

    added_risks = []
    for clause in added:
        label, score = predict_risk_label(clause)
        if label == "high risk":
            added_risks.append({
                "clause": clause,
                "risk_label": label,
                "risk_score": score,
            })

    modified_risk_deltas = []
    for item in modified:
        old_label, old_score = predict_risk_label(item["old_clause"])
        new_label, new_score = predict_risk_label(item["new_clause"])

        # Treat as risk increase if it flips to high risk, or stays high-risk with stronger confidence.
        is_increase = (old_label == "low risk" and new_label == "high risk") or (
            old_label == "high risk" and new_label == "high risk" and new_score > old_score
        )

        if is_increase:
            modified_risk_deltas.append({
                "old_clause": item["old_clause"],
                "new_clause": item["new_clause"],
                "similarity": item["similarity"],
                "old_risk_label": old_label,
                "old_risk_score": old_score,
                "new_risk_label": new_label,
                "new_risk_score": new_score,
                "diff": clause_word_diff(item["old_clause"], item["new_clause"]),
            })

    return {
        "added": added,
        "removed": removed,
        "modified": modified,
        "unchanged": unchanged,
        "structural_changes": structural_changes,
        "added_high_risk": added_risks,
        "modified_risk_increase": modified_risk_deltas,
        "diagnostics": diagnostics,
    }

In [ ]:
# print("nda1_clauses", nda1_clauses)
# print("nda2_clauses", nda2_clauses)

# Compare NDA-1 (old) vs NDA-2 (new)
version_report = analyze_version_changes(
    nda1_clauses,
    nda2_clauses,
    modified_threshold=0.58,
    candidate_floor=0.35,
    structural_threshold=0.42,
)

print("=== VERSION CHANGE SUMMARY ===")
print("Added clauses:", len(version_report["added"]))
print("Removed clauses:", len(version_report["removed"]))
print("Modified clauses:", len(version_report["modified"]))
print("Unchanged matched clauses:", len(version_report["unchanged"]))
print("Structural merge/split changes:", len(version_report["structural_changes"]))
print()
print("Newly added high-risk clauses:", len(version_report["added_high_risk"]))
print("Modified clauses with risk increase:", len(version_report["modified_risk_increase"]))

print("\n=== MATCHING DIAGNOSTICS ===")
print(version_report["diagnostics"])

print("\n=== ADDED HIGH-RISK CLAUSES (sample) ===")
for i, item in enumerate(version_report["added_high_risk"][:3], 1):
    print(f"[{i}] score={item['risk_score']:.3f}")
    print(item["clause"][:300], "\n")

print("\n=== STRUCTURAL CHANGES (sample) ===")
for i, item in enumerate(version_report["structural_changes"][:3], 1):
    print(f"[{i}] new clause:", item["new_clause"][:220])
    for j, linked in enumerate(item["linked_old_clauses"], 1):
        print(f"   linked old {j} sim={linked['similarity']:.3f}:", linked["old_clause"][:180])
    print()

print("\n=== MODIFIED CLAUSES WITH RISK INCREASE (sample) ===")
for i, item in enumerate(version_report["modified_risk_increase"][:3], 1):
    print(f"[{i}] similarity={item['similarity']:.3f}")
    print(f"old: {item['old_risk_label']} ({item['old_risk_score']:.3f})")
    print(f"new: {item['new_risk_label']} ({item['new_risk_score']:.3f})")
    print("old clause:", item["old_clause"][:220])
    print("new clause:", item["new_clause"][:220])
    print("word diff:")
    print(item["diff"][:1200])
    print()

=== VERSION CHANGE SUMMARY ===
Added clauses: 3
Removed clauses: 13
Modified clauses: 8
Unchanged matched clauses: 8
Structural merge/split changes: 0

Newly added high-risk clauses: 3
Modified clauses with risk increase: 2

=== MATCHING DIAGNOSTICS ===
{'matched_pairs': 16, 'candidate_pairs': 30, 'threshold': 0.2, 'candidate_floor': 0.35, 'structural_threshold': 0.42}

=== ADDED HIGH-RISK CLAUSES (sample) ===
[1] score=0.800
d) with anyone, including other students, outside of the course. 

[2] score=0.708
11. In the event of breach, Discloser may seek injunctive relief in addition to any other remedies available at law or in equity. 

[3] score=0.684
12. UT will ensure that access to Confidential Information is restricted to Students and instructional personnel with a need to know, and that such access is revoked at course completion. This Agreement is made effective on the Effective Date. By:______________________________________ Title:________ 


=== STRUCTURAL CHANGES (sample) ===